# Rich colours on Colab

A probe for one question: does Rich still emit colour once ShiroRVC is running
under Colab?

**Why it can fail.** The Start cell launches `app.py` with
`stdout=subprocess.PIPE` and reprints each line from the kernel. That makes the
server's `stderr` a pipe rather than a terminal, and Rich checks `isatty()` --
when it is false it strips every escape sequence. `rvc/lib/terminal.py` forces
the terminal back on when it detects Colab, and these cells check that it works.

Run the cells top to bottom. Each one states what it expects, so a wrong result
is readable on its own -- no need to compare against a reference screenshot.

## 1. Baseline

Before blaming Rich: can this output cell render ANSI at all? If these four
words come out grey, nothing downstream can help.

In [ ]:
print("\x1b[1;33mbold yellow\x1b[0m   \x1b[36mcyan\x1b[0m   "
      "\x1b[1;32mbold green\x1b[0m   \x1b[1;31mbold red\x1b[0m")
print("^ four coloured words expected above\n")

import os

for name in ("COLAB_RELEASE_TAG", "COLAB_GPU", "TERM", "COLORTERM",
             "FORCE_COLOR", "TTY_COMPATIBLE", "COLUMNS"):
    print(f"{name:18} = {os.environ.get(name)!r}")

## 2. Rich through the pipe

The same shape as the Start cell: a child process writes to a pipe, the kernel
reprints what comes back. It runs twice -- once with Rich left to autodetect,
once with `force_terminal=True`.

Expected: the first block colourless with `color_system=None`, the second
coloured. That difference is the whole bug, and the whole fix.

In [ ]:
import os
import pathlib
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rich"], check=True)

PROBE = pathlib.Path("/content/rich_probe.py")
PROBE.write_text('''
import os
import sys

from rich.console import Console

# None leaves Rich's autodetection alone; True is what terminal.py passes.
force = True if os.environ.get("PROBE_FORCE") == "1" else None
console = Console(
    file=sys.stderr,
    force_terminal=force,
    highlight=False,
    markup=False,
    soft_wrap=True,
)

print(
    f"    isatty={sys.stderr.isatty()}  is_terminal={console.is_terminal}  "
    f"color_system={console.color_system}  width={console.width}",
    file=sys.stderr,
)
console.print("    [EXTRACT] Config saved at logs/andre/config.json", style="cyan")
console.print("    [WARNING] mute audio missing", style="bold yellow")
console.print("    [OK] Pretrained model downloaded.", style="bold green")
''', encoding="utf-8")


def run_probe(label, force):
    # Launched the way the Start cell launches the server.
    print(f"--- {label}")
    env = {**os.environ, "PYTHONUNBUFFERED": "1",
           "PROBE_FORCE": "1" if force else "0"}
    process = subprocess.Popen(
        [sys.executable, str(PROBE)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env=env,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    process.wait()
    print()


run_probe("autodetect -> expect NO colour, color_system=None", force=False)
run_probe("force_terminal=True -> expect colour", force=True)

## 3. The repository's own console

Now the real thing: `rvc/lib/terminal.py`, down the same pipe. Its
`_force_terminal()` turns colour on only when it finds Colab in the environment,
so this probe runs twice -- once with `COLAB_RELEASE_TAG` stripped out, to
confirm the ordinary autodetection is still intact, and once with it present.

Expected: first block colourless, second coloured, with a cyan rule and a red
panel. Only Rich is needed, so the clone installs nothing.

In [ ]:
import os
import pathlib
import subprocess
import sys

# Deliberately not /content/ShiroRVC: that is where the Setup cell installs the
# app, and reusing it silently probes whatever revision happens to be sitting
# there. This clone is disposable and always refreshed.
ROOT = pathlib.Path("/content/ShiroRVC-probe")
REPO = "https://github.com/ShiromiyaG/ShiroRVC.git"
BRANCH = "main"  # point this at the branch that carries the fix

if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1",
                    "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-B", BRANCH,
                    "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO, str(ROOT)], check=True)

head = subprocess.run(["git", "-C", str(ROOT), "log", "-1", "--format=%h %s"],
                      capture_output=True, text=True).stdout.strip()
source = (ROOT / "rvc" / "lib" / "terminal.py").read_text(encoding="utf-8")
print(f"    HEAD: {head}")
# Without this the probe below is testing the unfixed code, and every result
# will read as "no colour" for the wrong reason.
print(f"    _force_terminal present: {'_force_terminal' in source}")
print()

PROBE = pathlib.Path("/content/terminal_probe.py")
PROBE.write_text('''
import sys

from rvc.lib.terminal import (
    error,
    get_console,
    info,
    print_error_panel,
    rule,
    success,
    warning,
)

console = get_console()
print(
    f"    isatty={sys.stderr.isatty()}  is_terminal={console.is_terminal}  "
    f"color_system={console.color_system}  width={console.width}",
    file=sys.stderr,
)

rule("Extraction")
info("Config saved at logs/andre/config.json", tag="[EXTRACT]")
success("Pretrained model downloaded.", tag="[DOWNLOAD]")
warning("mute audio missing, padding skipped", tag="[EXTRACT]")
error("something failed", tag="[EXTRACT]")
print_error_panel(
    "FileNotFoundError: logs/mute/sliced_audios/mute32000.wav",
    title="Extraction failed",
    details="preparing_files.py, line 66, in ensure_mute_audio",
)
''', encoding="utf-8")


def run_terminal_probe(label, colab):
    print(f"--- {label}")
    # The probe sits outside the clone, so sys.path[0] is /content; PYTHONPATH
    # is what makes `rvc` importable.
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONPATH": str(ROOT)}
    if colab:
        # Already present on a real runtime; set explicitly so the cell still
        # says something when it is run anywhere else.
        env.setdefault("COLAB_RELEASE_TAG", "probe")
    else:
        for name in ("COLAB_RELEASE_TAG", "COLAB_GPU", "FORCE_COLOR",
                     "TTY_COMPATIBLE"):
            env.pop(name, None)
    process = subprocess.Popen(
        [sys.executable, str(PROBE)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        cwd=str(ROOT),
        env=env,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    process.wait()
    print()


run_terminal_probe("Colab markers removed -> expect NO colour", colab=False)
run_terminal_probe("Colab markers present -> expect colour", colab=True)

## 4. Progress bars

Colour is one question; a bar that redraws in place is another. Rich animates a
progress bar with a carriage return plus `ESC[2K`, overwriting the same line --
never with newlines.

That makes the *reader* on the kernel side part of the experiment.
`subprocess.Popen(text=True)` turns on universal-newline translation, which
rewrites every lone carriage return as a newline. The bar's redraws become
separate lines before Colab ever sees them, and the animation arrives as a
stack. So this runs the same probe twice, changing only how the pipe is read.

Expected: the line reader stacks one frame per refresh with `CR=0`; the raw
passthrough reports a real `CR` count and redraws a single line in place. If the
second one still stacks, then Colab is the limit, not the reader.

The counts come from the bytes as they arrive, so they describe the stream
rather than what the cell chose to render.

In [ ]:
import codecs
import io
import os
import pathlib
import subprocess
import sys

ROOT = pathlib.Path("/content/ShiroRVC-probe")
if not ROOT.exists():
    raise SystemExit(f"{ROOT} is missing -- run the cell in section 3 first, "
                     "it is what clones the repository.")

PROBE = pathlib.Path("/content/progress_probe.py")
PROBE.write_text('''
import os
import time

from rvc.lib.terminal import track

# leave=False is the default, and it sets transient=True on the Progress --
# Rich erases the whole display when the bar completes.
leave = os.environ.get("PROBE_LEAVE") == "1"
for _ in track(range(40), description="Extracting features", leave=leave):
    time.sleep(0.05)
print("probe finished")
''', encoding="utf-8")

CR, LF, ESC = chr(13), chr(10), chr(27)


def run_progress_probe(label, raw):
    print(f"--- {label}")
    env = {
        **os.environ,
        "PYTHONUNBUFFERED": "1",
        "COLAB_RELEASE_TAG": "probe",
        "PYTHONPATH": str(ROOT),
        "PROBE_LEAVE": "1",
    }
    # No text=True here: the decoding is done below, so that each reader gets
    # the bytes exactly as the child wrote them.
    process = subprocess.Popen(
        [sys.executable, str(PROBE)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        cwd=str(ROOT),
        env=env,
    )
    seen = []
    if raw:
        # os.read hands back whatever has arrived, so a frame that ends in a
        # carriage return is not held until some later newline completes a line.
        decoder = codecs.getincrementaldecoder("utf-8")("replace")
        while True:
            data = os.read(process.stdout.fileno(), 4096)
            if not data:
                break
            text = decoder.decode(data)
            seen.append(text)
            sys.stdout.write(text)
            sys.stdout.flush()
    else:
        # Universal-newline translation is on by default, and it is what turns
        # every carriage return into a newline.
        stream = io.TextIOWrapper(process.stdout, encoding="utf-8",
                                  errors="replace")
        for line in stream:
            seen.append(line)
            print(line, end="", flush=True)
    status = process.wait()
    text = "".join(seen)
    print(f"    exit={status}  chars={len(text)}  "
          f"CR={text.count(CR)}  LF={text.count(LF)}  ESC={text.count(ESC)}")
    print()


run_progress_probe("line reader (text=True) -> carriage returns rewritten",
                   raw=False)
run_progress_probe("raw passthrough -> carriage returns preserved", raw=True)